# Telco Customer Churn Prediction using PySpark ML

## Project Idea
This project predicts whether a telecom customer will leave the company or stay.

**Target Column:** `Churn`

- `Yes` = Customer left the company
- `No` = Customer stayed with the company

## Main Goal
The goal is to apply Big Data concepts, Spark DataFrame analysis, machine learning, string column handling, and model evaluation.

# Section 1: Install and Import Libraries
This section installs PySpark and imports all libraries needed for Spark DataFrame operations, machine learning, and evaluation.

In [1]:
# Cell 1: Install PySpark
# This cell installs PySpark in Google Colab.
# PySpark is used for big data processing and machine learning.

!pip install pyspark -q

In [2]:
# Cell 2: Import Libraries
# This cell imports the required libraries.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, avg, round

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Section 2: Start Spark Session and Load Dataset
This section starts Spark and loads the Telco Customer Churn CSV file into a Spark DataFrame.

In [3]:
# Cell 3: Start Spark Session
# This cell creates a Spark session.

spark = SparkSession.builder \
    .appName("Telco Customer Churn Prediction") \
    .getOrCreate()

print("Spark session started successfully.")

Spark session started successfully.


In [4]:
# Cell 4: Upload Dataset
# This cell uploads the dataset from your computer to Google Colab.
# Upload the file: WA_Fn-UseC_-Telco-Customer-Churn.csv

from google.colab import files

uploaded = files.upload()

print("File uploaded successfully.")

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv
File uploaded successfully.


In [5]:
# Cell 5: Load CSV File
# This cell loads the uploaded CSV file into a Spark DataFrame.
# header=True means the first row contains column names.
# inferSchema=True allows Spark to detect data types automatically.

csv_file = list(uploaded.keys())[0]

df = spark.read.csv(
    csv_file,
    header=True,
    inferSchema=True
)

print("CSV file loaded successfully.")

df.show(5)
df.printSchema()

CSV file loaded successfully.
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|   

# Section 3: Dataset Overview and Cleaning
This section explores the dataset, checks missing values, and cleans the `TotalCharges` column.

In [6]:
# Cell 6: Dataset Overview
# This cell shows the number of rows and columns in the dataset.

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

print("Column names:")
print(df.columns)

Number of rows: 7043
Number of columns: 21
Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [7]:
# Cell 7: Check Missing Values
# This cell checks missing values in each column.

missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_values.show()

+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract|PaperlessBilling|PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|         0|     0|            0|      0|         0|     0|           0|            0|              0|             0|           0|               0|          0|          0|              0|       0|               0| 

In [8]:
# Cell 8: Clean TotalCharges Column
# This cell cleans the TotalCharges column.
# TotalCharges may contain blank spaces, so we convert them to null,
# then convert the column to double.

df = df.withColumn(
    "TotalCharges",
    when(col("TotalCharges") == " ", None).otherwise(col("TotalCharges"))
)

df = df.withColumn(
    "TotalCharges",
    col("TotalCharges").cast("double")
)

df = df.dropna()

print("Data cleaned successfully.")
df.printSchema()

Data cleaned successfully.
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



# Section 4: Data Analysis and Insights
This section applies Spark DataFrame operations such as `groupBy`, `count`, and `avg` to extract useful insights from the data.

In [9]:
# Cell 9: Count Churn and Non-Churn Customers
# This cell counts how many customers churned and how many did not.

print("Churn distribution:")

df.groupBy("Churn").count().show()

Churn distribution:
+-----+-----+
|Churn|count|
+-----+-----+
|   No| 5163|
|  Yes| 1869|
+-----+-----+



In [10]:
# Cell 10: Data Analysis 1 — Churn by Contract Type
# This cell analyzes churn based on contract type.

print("Churn by Contract Type:")

df.groupBy("Contract", "Churn") \
  .count() \
  .orderBy("Contract", "Churn") \
  .show()

Churn by Contract Type:
+--------------+-----+-----+
|      Contract|Churn|count|
+--------------+-----+-----+
|Month-to-month|   No| 2220|
|Month-to-month|  Yes| 1655|
|      One year|   No| 1306|
|      One year|  Yes|  166|
|      Two year|   No| 1637|
|      Two year|  Yes|   48|
+--------------+-----+-----+



In [11]:
# Cell 11: Data Analysis 2 — Average Monthly Charges by Churn
# This cell calculates the average monthly charges for churned and non-churned customers.

print("Average Monthly Charges by Churn:")

df.groupBy("Churn") \
  .agg(round(avg("MonthlyCharges"), 2).alias("Avg_MonthlyCharges")) \
  .show()

Average Monthly Charges by Churn:
+-----+------------------+
|Churn|Avg_MonthlyCharges|
+-----+------------------+
|   No|             61.31|
|  Yes|             74.44|
+-----+------------------+



In [12]:
# Cell 12: Data Analysis 3 — Churn by Internet Service
# This cell analyzes churn based on internet service type.

print("Churn by Internet Service:")

df.groupBy("InternetService", "Churn") \
  .count() \
  .orderBy("InternetService", "Churn") \
  .show()

Churn by Internet Service:
+---------------+-----+-----+
|InternetService|Churn|count|
+---------------+-----+-----+
|            DSL|   No| 1957|
|            DSL|  Yes|  459|
|    Fiber optic|   No| 1799|
|    Fiber optic|  Yes| 1297|
|             No|   No| 1407|
|             No|  Yes|  113|
+---------------+-----+-----+



In [13]:
# Cell 13: Data Analysis 4 — Churn by Payment Method
# This cell analyzes churn based on payment method.

print("Churn by Payment Method:")

df.groupBy("PaymentMethod", "Churn") \
  .count() \
  .orderBy("PaymentMethod", "Churn") \
  .show(truncate=False)

Churn by Payment Method:
+-------------------------+-----+-----+
|PaymentMethod            |Churn|count|
+-------------------------+-----+-----+
|Bank transfer (automatic)|No   |1284 |
|Bank transfer (automatic)|Yes  |258  |
|Credit card (automatic)  |No   |1289 |
|Credit card (automatic)  |Yes  |232  |
|Electronic check         |No   |1294 |
|Electronic check         |Yes  |1071 |
|Mailed check             |No   |1296 |
|Mailed check             |Yes  |308  |
+-------------------------+-----+-----+



In [14]:
# Cell 14: Data Analysis 5 — Average Tenure by Churn
# This cell calculates the average tenure for churned and non-churned customers.

print("Average Tenure by Churn:")

df.groupBy("Churn") \
  .agg(round(avg("tenure"), 2).alias("Avg_Tenure")) \
  .show()

Average Tenure by Churn:
+-----+----------+
|Churn|Avg_Tenure|
+-----+----------+
|   No|     37.65|
|  Yes|     17.98|
+-----+----------+



# Section 5: Prepare Data for Machine Learning
This section handles string columns using `StringIndexer` and `OneHotEncoder`, then combines all features into one vector column using `VectorAssembler`.

In [15]:
# Cell 15: Prepare Target Column
# This cell converts the target column Churn into a numeric label.
# Churn is the target column.

label_indexer = StringIndexer(
    inputCol="Churn",
    outputCol="label"
)

In [16]:
# Cell 16: Define Categorical and Numerical Columns
# Categorical columns are text columns that need encoding.
# Numerical columns can be used directly.

categorical_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

numerical_cols = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [17]:
# Cell 17: Handle String Columns using StringIndexer
# Machine learning models cannot understand text directly.
# StringIndexer converts text categories into numeric indexes.

indexers = [
    StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_index",
        handleInvalid="keep"
    )
    for col_name in categorical_cols
]

In [18]:
# Cell 18: Apply OneHotEncoder
# OneHotEncoder converts category indexes into binary vectors.
# This helps the ML model understand categorical columns better.

encoders = [
    OneHotEncoder(
        inputCol=col_name + "_index",
        outputCol=col_name + "_encoded"
    )
    for col_name in categorical_cols
]

In [19]:
# Cell 19: Combine Features using VectorAssembler
# Spark ML models require all input features in one vector column called features.

encoded_cols = [col_name + "_encoded" for col_name in categorical_cols]

assembler = VectorAssembler(
    inputCols=numerical_cols + encoded_cols,
    outputCol="features"
)

# Section 6: Build and Train Machine Learning Model
This section builds a Logistic Regression model and trains it using a Spark ML Pipeline.

In [20]:
# Cell 20: Build Logistic Regression Model
# Logistic Regression is used because Churn is a binary classification problem.

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction"
)

In [21]:
# Cell 21: Build ML Pipeline
# The pipeline includes:
# 1. Convert Churn into label
# 2. Convert string columns into numeric indexes
# 3. Apply OneHotEncoder
# 4. Combine features
# 5. Train Logistic Regression model

pipeline = Pipeline(
    stages=[label_indexer] + indexers + encoders + [assembler, lr]
)

In [22]:
# Cell 22: Split Data into Train and Test
# 80% is used for training and 20% is used for testing.

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Training data count:", train_df.count())
print("Testing data count:", test_df.count())

Training data count: 5690
Testing data count: 1342


In [23]:
# Cell 23: Train the Model
# This cell trains the machine learning model using the training data.

model = pipeline.fit(train_df)

print("Model trained successfully.")

Model trained successfully.


# Section 7: Prediction and Model Evaluation
This section makes predictions on unseen test data and evaluates the model using multiple metrics.

In [24]:
# Cell 24: Make Predictions
# This cell uses the trained model to predict customer churn on the test data.

predictions = model.transform(test_df)

predictions.select(
    "customerID",
    "Churn",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+----------+-----+-----+----------+-----------------------------------------+
|customerID|Churn|label|prediction|probability                              |
+----------+-----+-----+----------+-----------------------------------------+
|0004-TLHLJ|Yes  |1.0  |1.0       |[0.3470769151757115,0.6529230848242885]  |
|0013-SMEOE|No   |0.0  |0.0       |[0.9457645187950012,0.05423548120499877] |
|0015-UOCOJ|No   |0.0  |0.0       |[0.5662115983351705,0.43378840166482946] |
|0019-EFAEP|No   |0.0  |0.0       |[0.9530880143814118,0.046911985618588226]|
|0023-HGHWL|Yes  |1.0  |1.0       |[0.27346649354493907,0.7265335064550609] |
|0030-FNXPP|No   |0.0  |0.0       |[0.8049537292284737,0.19504627077152625] |
|0042-RLHYP|No   |0.0  |0.0       |[0.998814021245069,0.001185978754930983] |
|0057-QBUQH|No   |0.0  |0.0       |[0.9874113910582873,0.012588608941712653]|
|0078-XZMHT|No   |0.0  |0.0       |[0.9796901747177773,0.020309825282222693]|
|0080-EMYVY|No   |0.0  |0.0       |[0.8675908280982568,0.1324091

In [26]:
# Cell 25: Evaluate Model — Accuracy, F1, Precision, Recall
# This cell evaluates the model using classification metrics.

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1 = f1_evaluator.evaluate(predictions)
precision = precision_evaluator.evaluate(predictions)
recall = recall_evaluator.evaluate(predictions)

print("Accuracy:", format(accuracy, ".4f"))
print("F1 Score:", format(f1, ".4f"))
print("Precision:", format(precision, ".4f"))
print("Recall:", format(recall, ".4f"))

Accuracy: 0.8040
F1 Score: 0.7980
Precision: 0.7956
Recall: 0.8040


In [27]:
# Cell 26: Evaluate Model — AUC
# AUC measures how well the model separates churn and non-churn customers.

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(predictions)

print("AUC:", format(auc, ".4f"))

AUC: 0.8559


In [28]:
# Cell 27: Confusion Matrix
# This cell displays the confusion matrix.
# It shows correct and incorrect predictions for each class.

print("Confusion Matrix:")

predictions.groupBy("label", "prediction") \
    .count() \
    .orderBy("label", "prediction") \
    .show()

Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  886|
|  0.0|       1.0|  102|
|  1.0|       0.0|  161|
|  1.0|       1.0|  193|
+-----+----------+-----+

